# Inwiefern besteht ein Zusammenhang zwischen der Brutto CO2 Einspeisung, Trockenheit, Pflanzenmenge und der Bodentemperatur in Deutschland?

### bezogen auf die Hitzewelle 2018 in Deutschland

Was brauchen wir?

Erstmal: GPP Vergleich mit Trockenheit. Dafür: zwei Karten. Graph (Durschnittliche Trockenheit in Deutschland + Durschnittlicher GPP in Deutschland)

Plan::: Daten für GPP, Trockenheit, Bodentemperatur download. => auf gleiches Raster interpolieren. => alle drei auf nem Line Graph plotten

- CO2: CLMS_GPP_GLOBAL_300M_10DAILY_V2 2013-present
- Feuchtigkeit: CLMS_SSM_EUROPE_1KM_DAILY_V1 2014-present
- Temperatur: SENTINEL3_SLSTR_L2_LST 2016-present

=> 2016-present?

In [5]:
GPP_ID = "CLMS_GPP_GLOBAL_300M_10DAILY_V2"
SSM_ID = "CLMS_SSM_EUROPE_1KM_DAILY_V1"
LST_ID = "SENTINEL3_SLSTR_L2_LST"

In [6]:
spatial_extent = {
    'west': 5,
    'south': 46,
    'east': 15.5,
    'north': 56,
}

In [ ]:
temporal_extent = ["2017-01-01", "2019-12-31"]

Imports

In [8]:
import folium
import geopandas as gpd
import json
import leafmap
import math
import matplotlib.pyplot as plt
import os
import openeo
import pyproj
import rasterio

from rasterio.plot import show
from shapely.geometry import Polygon
from shapely import box

Authentification

In [9]:
from openeo.rest.auth.config import RefreshTokenStore

connection = openeo.connect("openeofed.dataspace.copernicus.eu")
connection.authenticate_oidc()
print(connection.describe_account())

Authenticated using refresh token.
{'info': {'oidc_userinfo': {'email': 'moritz.fechte@icloud.com', 'email_verified': True, 'family_name': 'Fe', 'given_name': 'Mo', 'name': 'Mo Fe', 'preferred_username': 'moritz.fechte@icloud.com', 'sub': '01b88e4b-35a5-4da2-8e10-edc6a6ce408f'}}, 'name': 'Mo Fe', 'user_id': '01b88e4b-35a5-4da2-8e10-edc6a6ce408f'}


### Datacube Preprocessing Methoden

Deutschland ausschneiden (germany.geojson Datei notwendig)

In [10]:
def mask_germany(datacube):
    germany = gpd.read_file("germany.geojson")
    geometry = germany.geometry.iloc[0].__geo_interface__

    return datacube.mask_polygon(
        geometry,
        srs="EPSG:4326"
    )

Zeit auf 10 Tage Intervall reduzieren

In [11]:
def reduce_to_decad(datacube):
    return datacube.aggregate_temporal_period(
        period="dekad",
        reducer="mean"
    )

Auflösung auf 1km reduzieren (resamplen)

In [12]:
def align_to_other_cube(grid_providing_cube, cube):
    return cube.resample_cube_spatial(
        target=grid_providing_cube,
        method="average"
    )

In [16]:
def get_mean(datacube, spatial_extent):
    geometry = box(
        spatial_extent["west"],
        spatial_extent["south"],
        spatial_extent["east"],
        spatial_extent["north"],
    ).__geo_interface__

    return datacube.aggregate_spatial(
        geometries=geometry,
        reducer='mean',
    )

### GPP Download

In [17]:
gpp = connection.load_collection(
    GPP_ID,
    spatial_extent=spatial_extent,
    temporal_extent = temporal_extent,
    bands=["gpp"],
)
gpp = gpp.band('gpp')

ssm = connection.load_collection(
    SSM_ID,
    spatial_extent=spatial_extent,
    temporal_extent=temporal_extent,
    bands=["ssm"],
)
ssm = ssm.band('ssm')

lst = connection.load_collection(
    LST_ID,
    spatial_extent=spatial_extent,
    temporal_extent=temporal_extent,
    bands=["LST"],
)
lst = lst.band('LST')

In [ ]:
# preprocessing
gpp = mask_germany(gpp)
gpp = align_to_other_cube(ssm, gpp)
gpp = reduce_to_decad(gpp)

ssm = mask_germany(ssm)
ssm = reduce_to_decad(ssm)

lst = mask_germany(lst)
lst = align_to_other_cube(ssm, lst)
lst = reduce_to_decad(lst)

gpp_means = get_mean(gpp, spatial_extent).execute()
ssm_means = get_mean(ssm, spatial_extent).execute()
lst_means = get_mean(lst, spatial_extent).execute()

# gpp.download('gpp.tif')